# 03 — Procesamiento Distribuido con Modin (Cloud — GCS)

Versión cloud del notebook Modin. Los CSVs se leen directamente desde **Google Cloud Storage** (`gs://big-data-proyecto-parcial/raw/`). Ray backend corre en el nodo Dataproc.

**Flujo:** GCS raw/ (2017–2025) → Modin+Ray (Dataproc node) → resultados por era COVID → BigQuery

| # | Operación | Descripción | Tabla BigQuery |
|---|-----------|-------------|----------------|
| 1 | **Create** | Tendencia anual de violencia doméstica por era COVID | `domestic_trend` |
| 2 | **Read**   | Crímenes por distrito y mes — comparativa PRE vs DURANTE vs POST | — (exploración) |
| 3 | **Read**   | Top 20 bloques más peligrosos con contexto de era | `top_blocks` |
| 4 | **Update** | Normalizar `location_description` + analizar cambio de espacios por COVID | — (enriquecimiento) |
| 5 | **Delete** | Eliminar duplicados por `case_number` y verificar integridad por era | — (limpieza) |

In [1]:
import subprocess
subprocess.run(['pip', 'install', 'gcsfs', 'modin[ray]', 'pandas-gbq', '--quiet'], check=True)
print('Dependencias cloud OK')

Dependencias cloud OK


In [2]:
import time
import modin.pandas as mpd
import pandas as pd
import ray
import gcsfs

PROJECT_ID = 'my-first-project-492901'
DATASET_ID = 'chicago_crimes_results'
GCS_BUCKET = 'gs://big-data-proyecto-parcial/raw'

YEARS_ANALYSIS = [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
ERA_MAP = {2017:'PRE', 2018:'PRE', 2019:'PRE',
           2020:'DURANTE', 2021:'DURANTE', 2022:'DURANTE',
           2023:'POST', 2024:'POST', 2025:'POST'}

def to_bigquery(df: pd.DataFrame, table_name: str, if_exists: str = 'replace') -> None:
    df.to_gbq(
        destination_table=f'{DATASET_ID}.{table_name}',
        project_id=PROJECT_ID,
        if_exists=if_exists,
        progress_bar=False,
    )
    print(f'  → BigQuery: {PROJECT_ID}.{DATASET_ID}.{table_name}  ({len(df):,} filas)')

ray.init(ignore_reinit_error=True)
print(f'Ray inicializado — CPUs: {ray.available_resources().get("CPU", "?")}')

2026-05-02 02:18:09,526	INFO worker.py:2012 -- Started a local Ray instance.


Ray inicializado — CPUs: 4.0


In [3]:
# ── Carga desde GCS via gcsfs (cambio clave vs. versión local) ────────────────
files = [f'{GCS_BUCKET}/Chicago_Crimes_{y}.csv' for y in YEARS_ANALYSIS]
GCS_OPTS = {"token": "google_default"}

t0 = time.time()
df = mpd.concat(
    [mpd.read_csv(f, low_memory=False, storage_options=GCS_OPTS) for f in files],
    ignore_index=True,
)
df['date']       = mpd.to_datetime(df['date'],       utc=True, errors='coerce')
df['updated_on'] = mpd.to_datetime(df['updated_on'], utc=True, errors='coerce')
df['covid_era']  = df['year'].map(ERA_MAP)

print(f'Fuente:  GCS — {GCS_BUCKET}/')
print(f'Cargado en {time.time()-t0:.1f}s')
print(f'Shape: {df.shape[0]:,} filas × {df.shape[1]} columnas')
print('Registros por era:')
for era in ['PRE', 'DURANTE', 'POST']:
    n = int((df['covid_era'] == era).sum())
    print(f'  {era:<8}: {n:>10,}')

Data types of partitions are different! Please refer to the troubleshooting section of the Modin documentation to fix this issue.


Fuente:  GCS — gs://big-data-proyecto-parcial/raw/
Cargado en 12.4s
Shape: 2,072,943 filas × 23 columnas
Registros por era:


  PRE     :    799,839
  DURANTE :    661,583
  POST    :    611,521


---
## CRUD 1 — CREATE: Tendencia anual de violencia doméstica por era COVID

In [4]:
df['arrest_bool']   = df['arrest'].astype(str).str.lower().eq('true')
df['domestic_bool'] = df['domestic'].astype(str).str.lower().eq('true')

domestic = df[df['domestic_bool']].copy()
domestic['year_quarter'] = (
    domestic['date'].dt.year.astype(str) + '-Q' + domestic['date'].dt.quarter.astype(str)
)

total_by_year    = df.groupby('year').size().reset_index(name='total_crimes')
domestic_by_year = domestic.groupby('year').size().reset_index(name='domestic_crimes')

result = total_by_year.merge(domestic_by_year, on='year', how='left').fillna(0)
result['domestic_crimes']   = result['domestic_crimes'].astype(int)
result['domestic_rate_pct'] = (result['domestic_crimes'] / result['total_crimes'] * 100).round(2)
result['year_quarter']      = result['year'].astype(str) + '-ANUAL'
result['covid_era']         = result['year'].map(ERA_MAP)
result = result[['year', 'covid_era', 'year_quarter', 'domestic_crimes', 'total_crimes', 'domestic_rate_pct']]

print('Tasa de violencia doméstica por era COVID:')
era_summary = result.groupby('covid_era').agg(
    domestic_crimes=('domestic_crimes', 'sum'),
    total_crimes=('total_crimes', 'sum'),
).reset_index()
era_summary['domestic_rate_pct'] = (era_summary['domestic_crimes'] / era_summary['total_crimes'] * 100).round(2)
for _, row in era_summary.iterrows():
    print(f'  {row["covid_era"]:<8}: {row["domestic_rate_pct"]:.2f}%  ({row["domestic_crimes"]:,} de {row["total_crimes"]:,})')

to_bigquery(result, 'domestic_trend')

Tasa de violencia doméstica por era COVID:
  DURANTE : 21.02%  (139,095 de 661,583)
  POST    : 18.39%  (112,471 de 611,521)


  PRE     : 19.20%  (153,534 de 799,839)


Please refer to https://modin.readthedocs.io/en/stable/supported_apis/defaulting_to_pandas.html for explanation.


  → BigQuery: my-first-project-492901.chicago_crimes_results.domestic_trend  (9 filas)


---
## CRUD 2 — READ: Crímenes por distrito y mes — comparativa PRE vs DURANTE vs POST

In [5]:
df['month'] = df['date'].dt.month
month_names = {1:'Ene',2:'Feb',3:'Mar',4:'Abr',5:'May',6:'Jun',
               7:'Jul',8:'Ago',9:'Sep',10:'Oct',11:'Nov',12:'Dic'}

print('Crímenes por distrito y mes — por era COVID (top 8 distritos):')
for era in ['PRE', 'DURANTE', 'POST']:
    agg = (
        df[df['covid_era'] == era]
          .dropna(subset=['district'])
          .groupby(['district', 'month'])
          .size()
          .reset_index(name='count')
    )
    import pandas as _pd
    agg_pd = _pd.DataFrame({
        'district': agg['district'].to_list(),
        'month':    agg['month'].to_list(),
        'count':    agg['count'].to_list(),
    })
    pivot_pd = agg_pd.pivot_table(index='district', columns='month', values='count', aggfunc='sum', fill_value=0)
    pivot_pd.columns = [month_names.get(int(c), c) for c in pivot_pd.columns]
    total_col = pivot_pd.sum(axis=1)
    top8 = pivot_pd.loc[total_col.nlargest(8).index]
    print(f'\n  Era {era}:')
    print(top8.to_string())

Crímenes por distrito y mes — por era COVID (top 8 distritos):



  Era PRE:
           Ene   Feb   Mar   Abr   May   Jun   Jul   Ago   Sep   Oct   Nov   Dic
district                                                                        
11.0      4531  3924  4531  4805  5249  4957  5251  5268  4656  4575  4155  4195
6.0       3836  3440  3764  4032  4616  4549  4794  4635  4268  4212  3920  3995
8.0       3985  3631  3878  3805  4252  4213  4478  4568  4128  4138  3856  3877
18.0      3617  3105  3371  3614  3986  4216  4373  4377  4034  4067  3795  3987
1.0       3505  3248  3620  3683  4112  4165  4144  4779  3768  3825  3761  3847
4.0       3293  2945  3419  3468  3914  3843  4083  3979  3748  3664  3246  3508
7.0       3156  2757  3190  3436  3993  3804  4152  4020  3628  3436  3183  3183
25.0      3492  2971  3215  3349  3528  3441  3737  3702  3434  3438  3117  3288



  Era DURANTE:
           Ene   Feb   Mar   Abr   May   Jun   Jul   Ago   Sep   Oct   Nov   Dic
district                                                                        
11.0      3218  3335  3628  3255  3879  3801  3839  3808  3738  3772  3268  3097
6.0       3334  3022  3558  3127  3721  3677  3902  3782  3731  3787  3477  3344
8.0       3297  2940  3207  2955  3462  3489  3654  3645  3632  3681  3498  3488
4.0       2998  2610  3029  2822  3189  3422  3643  3665  3541  3593  3063  3207
12.0      2580  2522  2656  2230  2783  3018  3185  3185  3385  3617  3281  3091
25.0      2606  2446  2700  2483  2938  2802  2987  3158  2995  3179  2994  2922
3.0       2756  2288  2649  2448  2950  2963  3066  3049  3045  2946  2713  2729
7.0       2647  2190  2551  2504  3125  3031  3092  3032  2813  2870  2475  2380



  Era POST:
           Ene   Feb   Mar   Abr   May   Jun   Jul   Ago   Sep   Oct   Nov   Dic
district                                                                        
8.0       3781  3676  3931  3778  4099  3245  3138  3017  3125  3041  2733  2634
12.0      3438  3148  3475  3645  3918  3200  2923  2917  2878  2861  2569  2478
6.0       3482  3188  3503  3546  3957  3018  2765  2652  2542  2664  2344  2252
1.0       3129  2932  3230  3289  3602  2946  2662  2972  2554  2528  2257  2250
4.0       3314  3062  3157  3400  3497  2821  2699  2445  2417  2592  2282  2127
11.0      3310  3020  3182  3185  3430  2947  2646  2547  2448  2285  2219  2202
19.0      2779  2623  2847  3062  3330  3090  2593  2677  2516  2440  2168  2106
25.0      3054  2802  2973  3124  3274  2622  2431  2409  2383  2400  2168  2144


---
## CRUD 3 — READ: Top 20 bloques más peligrosos con contexto de era

In [6]:
top_blocks = (
    df.groupby('block')
      .agg(
          total_crimes=('unique_key',        'count'),
          arrests=('arrest_bool',            'sum'),
          domestic_incidents=('domestic_bool', 'sum'),
          distinct_crime_types=('primary_type', 'nunique'),
      )
      .reset_index()
      .sort_values('total_crimes', ascending=False)
      .head(20)
)
top_blocks['arrest_rate_pct'] = (top_blocks['arrests'] / top_blocks['total_crimes'] * 100).round(2)
top_blocks = top_blocks.astype({
    'total_crimes': int, 'arrests': int,
    'domestic_incidents': int, 'distinct_crime_types': int,
})

era_per_block = (
    df.groupby(['block', 'covid_era'])
      .size()
      .reset_index(name='n')
      .sort_values('n', ascending=False)
      .drop_duplicates(subset='block', keep='first')[['block', 'covid_era']]
      .rename(columns={'covid_era': 'dominant_era'})
)
top_blocks = top_blocks.merge(era_per_block, on='block', how='left')

print('Top 10 bloques con más crímenes (2017–2025):')
print(top_blocks.head(10)[['block','total_crimes','arrests','arrest_rate_pct','distinct_crime_types','dominant_era']].to_string(index=False))

to_bigquery(
    top_blocks[['block','total_crimes','arrests','domestic_incidents','distinct_crime_types','arrest_rate_pct']],
    'top_blocks',
)

Top 10 bloques con más crímenes (2017–2025):
                              block  total_crimes  arrests  arrest_rate_pct  distinct_crime_types dominant_era
                   001XX N STATE ST          6257     2710            43.31                    24          PRE
                0000X W TERMINAL ST          3772     1514            40.14                    21          PRE
                   0000X N STATE ST          2944      717            24.35                    21          PRE
               008XX N MICHIGAN AVE          2305      624            27.07                    19          PRE
                   0000X S STATE ST          2152      600            27.88                    17          PRE
                   100XX W OHARE ST          2146      296            13.79                    20          PRE
                   011XX S CANAL ST          2131      333            15.63                    25          PRE
                033XX W FILLMORE ST          2031     1823         

  → BigQuery: my-first-project-492901.chicago_crimes_results.top_blocks  (20 filas)


---
## CRUD 4 — UPDATE: Normalizar `location_description` y analizar cambio de espacios por COVID

In [7]:
df['location_description'] = df['location_description'].str.strip().str.upper()

GROUP_MAP = {
    'STREET': 'VÍA PÚBLICA', 'SIDEWALK': 'VÍA PÚBLICA', 'ALLEY': 'VÍA PÚBLICA',
    'RESIDENCE': 'RESIDENCIAL', 'APARTMENT': 'RESIDENCIAL', 'HOUSE': 'RESIDENCIAL',
    'SMALL RETAIL STORE': 'COMERCIO', 'GROCERY FOOD STORE': 'COMERCIO',
    'RESTAURANT': 'COMERCIO', 'BAR OR TAVERN': 'COMERCIO',
    'SCHOOL': 'EDUCACIÓN', 'COLLEGE/UNIVERSITY': 'EDUCACIÓN',
    'CTA': 'TRANSPORTE', 'VEHICLE': 'TRANSPORTE', 'PARKING LOT': 'TRANSPORTE',
}

def map_group(loc):
    if pd.isna(loc):
        return 'DESCONOCIDO'
    for key, grp in GROUP_MAP.items():
        if key in str(loc):
            return grp
    return 'OTRO'

df['location_group'] = df['location_description'].apply(map_group)

loc_by_era = (
    df.groupby(['covid_era', 'location_group'])
      .size()
      .reset_index(name='count')
)
era_totals = loc_by_era.groupby('covid_era')['count'].transform('sum')
loc_by_era['pct'] = (loc_by_era['count'] / era_totals * 100).round(2)

print('Distribución de grupos de lugar por era COVID (%):')
pivot_loc = loc_by_era.pivot(index='location_group', columns='covid_era', values='pct')
if 'PRE' in pivot_loc.columns and 'DURANTE' in pivot_loc.columns:
    pivot_loc['ΔDURANTE-PRE'] = (pivot_loc['DURANTE'] - pivot_loc['PRE']).round(2)
print(pivot_loc[['PRE','DURANTE','POST','ΔDURANTE-PRE']].to_string())

Distribución de grupos de lugar por era COVID (%):
                  PRE  DURANTE   POST  ΔDURANTE-PRE
location_group                                     
COMERCIO         7.40     6.09   7.44         -1.31
DESCONOCIDO      0.47     0.60   0.42          0.13
EDUCACIÓN        1.98     1.14   1.58         -0.84
OTRO            17.08    12.66  13.12         -4.42
RESIDENCIAL     33.60    38.94  34.46          5.34
TRANSPORTE       7.77     6.92   7.50         -0.85
VÍA PÚBLICA     31.70    33.65  35.48          1.95


---
## CRUD 5 — DELETE: Eliminar duplicados por `case_number` y verificar integridad por era

In [8]:
import pandas as _pd
from collections import Counter

n_antes = len(df)
dup_mask   = df.duplicated(subset=['case_number'], keep=False)
era_list   = df['covid_era'].to_list()
dup_list   = dup_mask.to_list()

total_counter = Counter(era_list)
dup_counter   = Counter(e for e, d in zip(era_list, dup_list) if d)

dup_report = _pd.DataFrame({
    'total':      [total_counter.get(e, 0) for e in ['PRE','DURANTE','POST']],
    'duplicados': [dup_counter.get(e, 0)   for e in ['PRE','DURANTE','POST']],
}, index=['PRE','DURANTE','POST'])
dup_report['pct_dup'] = (dup_report['duplicados'] / dup_report['total'] * 100).round(3)

total_dups = sum(dup_list)
print(f'Registros totales:    {n_antes:,}')
print(f'Registros duplicados: {total_dups:,}  ({total_dups/n_antes*100:.3f}%)')
print('\nDuplicados por era COVID:')
print(dup_report.to_string())

df_dedup  = df.drop_duplicates(subset=['case_number'], keep='first')
n_despues = len(df_dedup)
print(f'\nRegistros únicos:     {n_despues:,}')
print(f'Filas eliminadas:     {n_antes - n_despues:,}')

ray.shutdown()
print('Ray cerrado. Resultados guardados en BigQuery.')

Registros totales:    2,072,943
Registros duplicados: 494  (0.024%)

Duplicados por era COVID:
          total  duplicados  pct_dup
PRE      799839         169    0.021
DURANTE  661583         209    0.032
POST     611521         116    0.019



Registros únicos:     2,072,679
Filas eliminadas:     264


Ray cerrado. Resultados guardados en BigQuery.
